# Audio Analysis Master Notebook

Reusable, configuration-driven workflow for locally supplied video or audio files. Set `INPUT_MEDIA` in the configuration cell, then run the preparation and analysis cells in order. Local videos are converted to a WAV analysis track with FFmpeg; no downloading is performed. Derived files are written under `results/master/<input-stem>/`. Optional MERT and Basic Pitch cells require their own dependencies.

In [ ]:
!sudo apt-get install -y ffmpeg

In [ ]:
%pip install librosa scikit-learn demucs luigi

# Local media setup and Demucs stem separation


In [ ]:
from pathlib import Path
import os
import re
import shutil
import subprocess
import soundfile as sf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
MEDIA_SOURCE_DIR = Path(os.environ.get("MEDIA_SOURCE_DIR", PROJECT_ROOT / "video")).expanduser().resolve()
MASTER_RESULTS_ROOT = PROJECT_ROOT / "results" / "master"
INPUT_MEDIA = MEDIA_SOURCE_DIR / "-国庆111572550806.mp4"  # Replace with the file in video/.

if not INPUT_MEDIA.is_file():
    raise FileNotFoundError(f"Set INPUT_MEDIA to an existing local media file: {INPUT_MEDIA}")
if shutil.which("ffmpeg") is None:
    raise FileNotFoundError("FFmpeg is required for local media audio preparation.")

MASTER_RESULTS_DIR = MASTER_RESULTS_ROOT / INPUT_MEDIA.stem
MASTER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
video_suffixes = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}
audio_suffixes = {".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".aiff", ".aif"}
if INPUT_MEDIA.suffix.lower() in video_suffixes:
    SOURCE_AUDIO = MASTER_RESULTS_DIR / "audio_original.wav"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(INPUT_MEDIA), "-vn",
        "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(SOURCE_AUDIO),
    ], check=True)
elif INPUT_MEDIA.suffix.lower() in audio_suffixes:
    SOURCE_AUDIO = INPUT_MEDIA
else:
    raise ValueError(f"Unsupported local media extension: {INPUT_MEDIA.suffix}")

# Detect sustained trailing silence; preserve the full source and analyze a trimmed copy.
TRIM_TRAILING_SILENCE = True
SILENCE_THRESHOLD_DB = -40
MIN_TRAILING_SILENCE_SECONDS = 0.5
SILENCE_DETECT_MIN_SECONDS = 0.25
source_duration_s = float(sf.info(SOURCE_AUDIO).duration)
detection = subprocess.run([
    "ffmpeg", "-hide_banner", "-i", str(SOURCE_AUDIO),
    "-af", f"silencedetect=noise={SILENCE_THRESHOLD_DB}dB:d={SILENCE_DETECT_MIN_SECONDS}",
    "-f", "null", "-",
], check=True, capture_output=True, text=True)
events = re.findall(r"silence_(start|end):\s*([0-9.]+)", detection.stderr)
open_silence_start = None
trailing_silence_start = None
for event, time_value in events:
    event_time = float(time_value)
    if event == "start":
        open_silence_start = event_time
    elif open_silence_start is not None:
        if event_time >= source_duration_s - 0.1:
            trailing_silence_start = open_silence_start
        open_silence_start = None
if open_silence_start is not None and open_silence_start < source_duration_s:
    trailing_silence_start = open_silence_start

INPUT_AUDIO = SOURCE_AUDIO
if TRIM_TRAILING_SILENCE and trailing_silence_start is not None:
    silence_duration_s = source_duration_s - trailing_silence_start
    if silence_duration_s >= MIN_TRAILING_SILENCE_SECONDS:
        if trailing_silence_start <= 0:
            raise ValueError("Silence detector found no usable audio before the end of the source.")
        INPUT_AUDIO = MASTER_RESULTS_DIR / "audio_analysis_trimmed.wav"
        subprocess.run([
            "ffmpeg", "-y", "-i", str(SOURCE_AUDIO), "-t", f"{trailing_silence_start:.6f}",
            "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(INPUT_AUDIO),
        ], check=True)
        print(f"Detected {silence_duration_s:.3f}s trailing silence; analysis copy ends at {trailing_silence_start:.3f}s.")
    else:
        print(f"Trailing silence ({silence_duration_s:.3f}s) is shorter than the configured trim minimum; keeping full audio.")
else:
    print("No qualifying trailing silence detected; keeping full audio.")

DEMUX_OUTPUT_ROOT = MASTER_RESULTS_DIR / "demucs"
DEMUX_STEM_DIR = DEMUX_OUTPUT_ROOT / "htdemucs" / INPUT_AUDIO.stem
files = {
    "original_mix": INPUT_AUDIO,
    "vocals": DEMUX_STEM_DIR / "vocals.wav",
    "no_vocals": DEMUX_STEM_DIR / "no_vocals.wav",
}
output_dir = MASTER_RESULTS_DIR
print(f"Local source: {INPUT_MEDIA}")
print(f"Preserved source audio: {SOURCE_AUDIO}")
print(f"Analysis audio: {INPUT_AUDIO}")
print(f"Results: {MASTER_RESULTS_DIR}")

In [ ]:
# Separate the configured input into vocals and accompaniment.
import subprocess
import sys

if not INPUT_AUDIO.is_file():
    raise FileNotFoundError(f"Input audio not found: {INPUT_AUDIO}")
DEMUX_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, "-m", "demucs",
    "-n", "htdemucs",
    "--two-stems=vocals",
    "-o", str(DEMUX_OUTPUT_ROOT),
    str(INPUT_AUDIO),
], check=True)

missing = [str(path) for path in files.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Demucs did not create expected audio file(s): " + ", ".join(missing))
print("Created audio variants:")
for part, path in files.items():
    print(f"{part}: {path}")

## Whole-track summaries

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa

files = {
    "original_mix": INPUT_AUDIO,
    "vocals": DEMUX_STEM_DIR / "vocals.wav",
    "no_vocals": DEMUX_STEM_DIR / "no_vocals.wav",
}

pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
summary, chroma_rows = [], []

for part, path in files.items():
    if not path.exists():
        print(f"Skipping missing file: {path}")
        continue

    y, sr = librosa.load(path, sr=None, mono=True)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    tempo = librosa.feature.tempo(y=y, sr=sr)[0]

    summary.append({
        "part": part,
        "file": str(path),
        "duration_s": round(len(y) / sr, 2),
        "sample_rate": sr,
        "rms_mean": float(librosa.feature.rms(y=y).mean()),
        "tempo_bpm_estimate": float(tempo),
        "spectral_contrast_mean": float(contrast.mean()),
        **{f"mfcc_{i+1}_mean": float(mfcc[i].mean()) for i in range(13)},
    })
    chroma_rows.extend(
        {"part": part, "pitch_class": note, "mean_chroma": float(chroma[i].mean())}
        for i, note in enumerate(pitch_classes)
    )

results = MASTER_RESULTS_DIR
results.mkdir(parents=True, exist_ok=True)
pd.DataFrame(summary).to_csv(results / "audio_feature_summary.csv", index=False)
pd.DataFrame(chroma_rows).to_csv(results / "audio_chroma_profile.csv", index=False)

pd.DataFrame(summary)

## Audio manifest and provenance

Create one traceable record per input variant. Fill in source URL and retrieval time when applicable; the source-file and per-variant checksums provide stable identifiers.

In [ ]:
import pandas as pd
import soundfile as sf

from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import version

def file_sha256(path, chunk_size=1024 * 1024):
    digest = sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_media_sha256 = file_sha256(INPUT_MEDIA)
source_audio_sha256 = file_sha256(SOURCE_AUDIO)
analysis_audio_sha256 = file_sha256(INPUT_AUDIO)
sample_id = source_media_sha256[:16]
manifest_generated_at = datetime.now(timezone.utc).isoformat()
demucs_version = version("demucs")
manifest_rows = []

for stem_name, stem_path in files.items():
    info = sf.info(stem_path)
    is_original = stem_name == "original_mix"
    manifest_rows.append({
        "sample_id": sample_id,
        "title": INPUT_MEDIA.stem,
        "source_url": "",
        "search_query": "",
        "retrieved_at_utc": "",
        "manifest_generated_at_utc": manifest_generated_at,
        "source_path": str(INPUT_MEDIA.resolve()),
        "source_media_sha256": source_media_sha256,
        "analysis_audio_path": str(INPUT_AUDIO.resolve()),
        "source_audio_sha256": source_audio_sha256,
        "analysis_audio_sha256": analysis_audio_sha256,
        "audio_variant": stem_name,
        "audio_path": str(stem_path.resolve()),
        "parent_audio_path": "" if is_original else str(INPUT_AUDIO.resolve()),
        "separation_model": "" if is_original else "htdemucs",
        "separation_target": "" if is_original else "vocals",
        "separation_parameters": "" if is_original else "--two-stems=vocals",
        "demucs_version": demucs_version if not is_original else "",
        "duration_seconds": float(info.duration),
        "sample_rate_hz": int(info.samplerate),
        "channels": int(info.channels),
        "audio_file_size_bytes": int(stem_path.stat().st_size),
        "audio_sha256": file_sha256(stem_path),
        "feature_window_seconds": 5.0,
        "feature_hop_seconds": 2.5,
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = MASTER_RESULTS_DIR / "audio_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"Saved {len(manifest_df)} audio records to {manifest_path}")
print(manifest_df[["sample_id", "audio_variant", "audio_path", "duration_seconds", "sample_rate_hz", "channels"]].to_string(index=False))

## Audio quality checks

Check file metadata, clipping, near-silence, RMS, and waveform for each variant. These are technical diagnostics, not content labels.

In [ ]:
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt

quality_dir = MASTER_RESULTS_DIR / "quality"
quality_dir.mkdir(parents=True, exist_ok=True)
audio_variants = {}
quality_rows = []
silence_threshold = 1e-4

for stem_name, stem_path in files.items():
    audio_info = sf.info(stem_path)
    y, sr = librosa.load(stem_path, sr=None, mono=True)
    duration_s = len(y) / sr
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    frame_rms = librosa.feature.rms(y=y)[0]
    silence_fraction = float(np.mean(frame_rms < silence_threshold)) if len(frame_rms) else 0.0
    quality = {
        "stem": stem_name,
        "file": str(stem_path.resolve()),
        "duration_seconds": duration_s,
        "sample_rate_hz": int(audio_info.samplerate),
        "channels": int(audio_info.channels),
        "frames": int(audio_info.frames),
        "peak_absolute_amplitude": peak,
        "clipped_sample_fraction": float(np.mean(np.abs(y) >= 0.999)) if len(y) else 0.0,
        "rms_mean": float(np.sqrt(np.mean(y ** 2))) if len(y) else 0.0,
        "near_silence_frame_fraction": silence_fraction,
        "near_silence_rms_threshold": silence_threshold,
    }
    (quality_dir / f"audio_quality_{stem_name}.json").write_text(
        json.dumps(quality, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    time = np.arange(len(y)) / sr
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(time, y, linewidth=0.35)
    ax.set(xlabel="Time (seconds)", ylabel="Amplitude", title=f"Waveform: {stem_name}")
    fig.tight_layout()
    fig.savefig(quality_dir / f"audio_waveform_{stem_name}.png", dpi=150)
    plt.show()
    audio_variants[stem_name] = {"path": stem_path, "y": y, "sr": sr, "duration_s": duration_s}
    quality_rows.append(quality)

quality_df = pd.DataFrame(quality_rows)
display(quality_df)
quality_csv = quality_dir / "audio_quality_summary.csv"
quality_df.to_csv(quality_csv, index=False)
print("Saved quality summary to:", quality_csv)


## Windowed handcrafted features

Extract the same 62 interpretable features from each 5-second window with a 2.5-second hop. The window rows retain variant and timestamps for later comparison or clustering.

In [ ]:
window_seconds = 5.0
hop_seconds = 2.5
min_tail_seconds = 0.5
all_feature_frames = []

def summarize_feature(name, values, row):
    values = np.asarray(values, dtype=float)
    row[f"{name}_mean"] = float(np.mean(values))
    row[f"{name}_std"] = float(np.std(values))

for stem_name, variant in audio_variants.items():
    y = variant["y"]
    sr = variant["sr"]
    duration_s = variant["duration_s"]
    feature_rows = []
    for start_s in np.arange(0.0, duration_s, hop_seconds):
        end_s = min(start_s + window_seconds, duration_s)
        if end_s - start_s < min_tail_seconds:
            continue
        start_sample = int(round(start_s * sr))
        end_sample = min(int(round(end_s * sr)), len(y))
        segment = y[start_sample:end_sample]
        if len(segment) == 0:
            continue

        row = {"stem": stem_name, "start_seconds": float(start_s), "end_seconds": float(end_s)}
        mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
        for i in range(mfcc.shape[0]):
            row[f"mfcc_{i + 1}_mean"] = float(np.mean(mfcc[i]))
            row[f"mfcc_{i + 1}_std"] = float(np.std(mfcc[i]))

        chroma = librosa.feature.chroma_stft(y=segment, sr=sr)
        for i in range(chroma.shape[0]):
            row[f"chroma_{i}"] = float(np.mean(chroma[i]))

        summarize_feature("rms", librosa.feature.rms(y=segment)[0], row)
        summarize_feature("spectral_centroid", librosa.feature.spectral_centroid(y=segment, sr=sr)[0], row)
        summarize_feature("spectral_bandwidth", librosa.feature.spectral_bandwidth(y=segment, sr=sr)[0], row)
        summarize_feature("spectral_rolloff", librosa.feature.spectral_rolloff(y=segment, sr=sr)[0], row)
        summarize_feature("zero_crossing_rate", librosa.feature.zero_crossing_rate(y=segment)[0], row)
        contrast = librosa.feature.spectral_contrast(y=segment, sr=sr)
        for i in range(contrast.shape[0]):
            row[f"spectral_contrast_{i}_mean"] = float(np.mean(contrast[i]))
            row[f"spectral_contrast_{i}_std"] = float(np.std(contrast[i]))
        feature_rows.append(row)

    stem_df = pd.DataFrame(feature_rows)
    feature_count = len([name for name in stem_df.columns if name not in {"stem", "start_seconds", "end_seconds"}])
    if feature_count != 62:
        raise ValueError(f"Expected 62 numeric features for {stem_name}, found {feature_count}")
    stem_feature_path = MASTER_RESULTS_DIR / f"handcrafted_audio_vectors_{stem_name}.csv"
    stem_df.to_csv(stem_feature_path, index=False)
    all_feature_frames.append(stem_df)
    print(f"{stem_name}: {len(stem_df)} windows x {stem_df.shape[1] - 3} features -> {stem_feature_path}")

feature_df = pd.concat(all_feature_frames, ignore_index=True)
combined_feature_path = MASTER_RESULTS_DIR / "handcrafted_audio_vectors_all_stems.csv"
feature_df.to_csv(combined_feature_path, index=False)
print("Combined rows:", len(feature_df))
print("Feature dimensions (excluding stem and timestamps):", feature_df.shape[1] - 3)
print("Saved combined table:", combined_feature_path)
print(feature_df.groupby("stem").size())
print(feature_df.head().to_string(index=False))

## Multi-variant music analysis

Run beat tracking, chroma/key candidates, pitch-class timelines, coarse structure, recurrence plots, and original-to-stem chroma DTW for every configured variant. These estimates require listening and human review.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import display

music_out_dir = MASTER_RESULTS_DIR / "music_analysis"
music_out_dir.mkdir(parents=True, exist_ok=True)
analysis_sr = 22050
chroma_hop = 2048
pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
major_template = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
minor_template = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
chroma_by_stem = {}
music_summary_rows = []

for stem_name, variant in audio_variants.items():
    y, sr = librosa.load(variant["path"], sr=analysis_sr, mono=True)
    stem_dir = music_out_dir / stem_name
    stem_dir.mkdir(parents=True, exist_ok=True)
    duration_s = len(y) / sr

    # Beat tracking, plus intervals between detected beats as a local tempo cue.
    beat_hop = 512
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=beat_hop)
    tempo, beat_frames = librosa.beat.beat_track(
        onset_envelope=onset_env, sr=sr, hop_length=beat_hop, units="frames"
    )
    beat_times = librosa.frames_to_time(beat_frames, sr=sr, hop_length=beat_hop)
    tempo_bpm = float(np.asarray(tempo).reshape(-1)[0])
    beat_intervals = np.diff(beat_times)
    local_bpm = np.divide(
        60.0, beat_intervals, out=np.full(beat_intervals.shape, np.nan), where=beat_intervals > 0
    )
    beat_df = pd.DataFrame({
        "beat_number": np.arange(1, len(beat_times) + 1),
        "time_seconds": beat_times,
        "interval_from_previous_seconds": np.r_[np.nan, beat_intervals],
        "local_bpm": np.r_[np.nan, local_bpm],
    })
    beat_df.to_csv(stem_dir / "beat_times.csv", index=False)
    onset_times = librosa.frames_to_time(np.arange(len(onset_env)), sr=sr, hop_length=beat_hop)
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(onset_times, onset_env, linewidth=0.7, label="Onset strength")
    if len(beat_times):
        ax.scatter(beat_times, np.interp(beat_times, onset_times, onset_env), s=14, color="tab:red", label="Detected beats")
    ax.set(title=f"Beat tracking: {stem_name} ({tempo_bpm:.1f} BPM estimate)", xlabel="Time (seconds)", ylabel="Onset strength")
    ax.legend()
    fig.tight_layout()
    fig.savefig(stem_dir / "beat_tracking.png", dpi=140)
    plt.show()

    # Chroma profile, heuristic key candidates, and five-second pitch-class timeline.
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=chroma_hop)
    chroma_by_stem[stem_name] = chroma
    chroma_profile = chroma.mean(axis=1)
    key_rows = []
    for mode, template in [("maj", major_template), ("min", minor_template)]:
        for tonic_index, tonic in enumerate(pitch_classes):
            correlation = np.corrcoef(chroma_profile, np.roll(template, tonic_index))[0, 1]
            key_rows.append({
                "key": f"{tonic}:{mode}", "tonic": tonic, "mode": mode,
                "profile_correlation": float(correlation),
            })
    key_df = pd.DataFrame(key_rows).sort_values("profile_correlation", ascending=False).reset_index(drop=True)
    key_df.to_csv(stem_dir / "key_candidates.csv", index=False)
    pd.DataFrame({"pitch_class": pitch_classes, "mean_chroma": chroma_profile}).to_csv(
        stem_dir / "chroma_profile.csv", index=False
    )
    frame_times = librosa.frames_to_time(np.arange(chroma.shape[1]), sr=sr, hop_length=chroma_hop)
    window_ids = np.floor(frame_times / 5.0).astype(int)
    timeline_rows = []
    for window_id in np.unique(window_ids):
        in_window = window_ids == window_id
        window_chroma = chroma[:, in_window]
        dominant_index = int(window_chroma.mean(axis=1).argmax())
        timeline_rows.append({
            "start_seconds": float(window_id * 5.0),
            "end_seconds": float(min((window_id + 1) * 5.0, duration_s)),
            "dominant_pitch_class": pitch_classes[dominant_index],
            "mean_chroma": float(window_chroma[dominant_index].mean()),
        })
    pd.DataFrame(timeline_rows).to_csv(stem_dir / "pitch_class_timeline_5s.csv", index=False)

    # Coarse chroma-based boundaries, with a complete interval table including audio end.
    n_frames = chroma.shape[1]
    boundary_frames = librosa.segment.agglomerative(chroma, k=min(8, n_frames))
    candidate_times = librosa.frames_to_time(boundary_frames, sr=sr, hop_length=chroma_hop)
    pd.DataFrame({"boundary_frame": boundary_frames, "time_seconds": candidate_times}).to_csv(
        stem_dir / "candidate_segment_boundaries.csv", index=False
    )
    interval_frames = np.unique(np.r_[0, boundary_frames, n_frames])
    interval_times = librosa.frames_to_time(interval_frames, sr=sr, hop_length=chroma_hop)
    interval_times[0] = 0.0
    interval_times[-1] = duration_s
    pd.DataFrame({
        "segment_id": np.arange(1, len(interval_times)),
        "start_seconds": interval_times[:-1],
        "end_seconds": interval_times[1:],
    }).to_csv(stem_dir / "candidate_segment_intervals.csv", index=False)
    recurrence = librosa.segment.recurrence_matrix(chroma, mode="affinity", self=True, width=3)
    fig, ax = plt.subplots(figsize=(6, 5))
    librosa.display.specshow(recurrence, x_axis="time", y_axis="time", sr=sr, hop_length=chroma_hop, cmap="magma", ax=ax)
    ax.set_title(f"Chroma self-similarity: {stem_name}")
    fig.tight_layout()
    fig.savefig(stem_dir / "recurrence_matrix.png", dpi=140)
    plt.show()

    # Save a compact chroma plot for visual review.
    fig, ax = plt.subplots(figsize=(14, 4))
    img = librosa.display.specshow(chroma, x_axis="time", y_axis="chroma", sr=sr, hop_length=chroma_hop, ax=ax)
    ax.set_title(f"CQT chroma: {stem_name}")
    fig.colorbar(img, ax=ax)
    fig.tight_layout()
    fig.savefig(stem_dir / "chroma_timeline.png", dpi=140)
    plt.show()

    best = key_df.iloc[0]
    margin = float(best["profile_correlation"] - key_df.iloc[1]["profile_correlation"])
    music_summary_rows.append({
        "stem": stem_name, "duration_seconds": duration_s, "estimated_tempo_bpm": tempo_bpm,
        "detected_beats": len(beat_times), "top_key_candidate": best["key"],
        "top_key_correlation": float(best["profile_correlation"]), "top_key_margin": margin,
        "candidate_boundaries": len(boundary_frames),
    })

music_summary = pd.DataFrame(music_summary_rows)
music_summary.to_csv(music_out_dir / "music_summary.csv", index=False)
display(music_summary)

# Compare original mix with each Demucs stem using normalized chroma DTW.
alignment_rows = []
reference_name = "original_mix"
for candidate_name in ["vocals", "no_vocals"]:
    X = librosa.util.normalize(chroma_by_stem[reference_name], axis=0)
    Y = librosa.util.normalize(chroma_by_stem[candidate_name], axis=0)
    D, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
    alignment_cost = float(D[-1, -1] / max(len(wp), 1))
    alignment_df = pd.DataFrame({
        "original_frame": wp[::-1, 0], "candidate_frame": wp[::-1, 1],
        "original_time_seconds": librosa.frames_to_time(wp[::-1, 0], sr=analysis_sr, hop_length=chroma_hop),
        "candidate_time_seconds": librosa.frames_to_time(wp[::-1, 1], sr=analysis_sr, hop_length=chroma_hop),
    })
    alignment_df.to_csv(music_out_dir / f"dtw_original_vs_{candidate_name}.csv", index=False)
    alignment_rows.append({"reference": reference_name, "candidate": candidate_name, "normalized_dtw_cost": alignment_cost})
pd.DataFrame(alignment_rows).to_csv(music_out_dir / "dtw_summary.csv", index=False)
display(pd.DataFrame(alignment_rows))
print("Saved music analysis to:", music_out_dir)

## Windowed music vectors: handcrafted baseline and optional MERT

The summary features above are mostly whole-track aggregates. For comparing musical passages, create one fixed-size vector per timestamped window. The test-video notebook demonstrates this with 5-second windows and a 2.5-second hop across original/vocals/no_vocals stems. Its 62 handcrafted values are an interpretable baseline (MFCC, chroma, energy, spectral statistics), not a validated optimal feature set.

For a learned music representation, the optional cell below encodes each available track/stem window with `m-a-p/MERT-v1-95M` (5-second context, 24 kHz input) and mean-pools the last hidden state. First install compatible `transformers` and PyTorch in this notebook kernel. Running `from_pretrained` downloads model code/weights; `trust_remote_code=True` executes repository code, so verify its source, license, and code before running. This cell is optional; it does not run unless explicitly executed.

The output is a matrix plus an index with the source path and time span for every vector. MERT embeddings, handcrafted features, key/tempo summaries, and Basic Pitch note events are different representations: compare them separately before considering fusion. For a large corpus, retain the same model, sample rate, windowing, and pooling across files; inspect nearest neighbors by listening. These vectors express model/feature-space similarity, not a music label or propaganda judgment.

In [ ]:
# Optional: run after the `files` mapping is defined in Basic summaries.
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import AutoModel, Wav2Vec2FeatureExtractor

MERT_MODEL_ID = "m-a-p/MERT-v1-95M"
mert_processor = Wav2Vec2FeatureExtractor.from_pretrained(
    MERT_MODEL_ID, trust_remote_code=True
)
mert_model = AutoModel.from_pretrained(
    MERT_MODEL_ID, trust_remote_code=True
)
mert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mert_model = mert_model.to(mert_device).eval()
mert_sr = int(mert_processor.sampling_rate)
window_seconds, hop_seconds = 5.0, 2.5
min_tail_seconds = 0.5
embedding_rows, embedding_vectors = [], []

for part, path in files.items():
    if not path.is_file():
        raise FileNotFoundError(f"Audio file for {part!r} not found: {path}")
    audio, source_sr = librosa.load(path, sr=None, mono=True)
    duration_s = len(audio) / source_sr
    audio = librosa.resample(audio, orig_sr=source_sr, target_sr=mert_sr).astype(np.float32)
    window_samples = int(round(window_seconds * mert_sr))

    for start_s in np.arange(0.0, duration_s, hop_seconds):
        end_s = min(start_s + window_seconds, duration_s)
        actual_duration_s = end_s - start_s
        if actual_duration_s < window_seconds:
            continue
        start_sample = int(round(start_s * mert_sr))
        end_sample = min(int(round(end_s * mert_sr)), len(audio))
        segment = audio[start_sample:end_sample]
        padded_samples = 0

        inputs = mert_processor(segment, sampling_rate=mert_sr, return_tensors="pt")
        inputs = {name: value.to(mert_device) for name, value in inputs.items()}
        with torch.inference_mode():
            outputs = mert_model(**inputs)
            hidden_state = getattr(outputs, "last_hidden_state", None)
            if hidden_state is None:
                hidden_states = getattr(outputs, "hidden_states", None)
                if not hidden_states:
                    raise RuntimeError("MERT returned no last_hidden_state or hidden_states")
                hidden_state = hidden_states[-1]
            vector = hidden_state.mean(dim=1).squeeze(0).cpu().numpy().astype(np.float32)

        embedding_vectors.append(vector)
        embedding_rows.append({
            "embedding_row": len(embedding_rows),
            "part": part,
            "source_path": str(path),
            "start_seconds": float(start_s),
            "end_seconds": float(end_s),
            "actual_duration_seconds": float(actual_duration_s),
            "padded_duration_seconds": float(padded_samples / mert_sr),
            "model_id": MERT_MODEL_ID,
            "sample_rate_hz": mert_sr,
            "pooling": "last_hidden_state_mean_over_time",
        })

if not embedding_vectors:
    raise RuntimeError("No audio windows were encoded; check input files and durations")

mert_matrix = np.stack(embedding_vectors)
mert_output_dir = MASTER_RESULTS_DIR / "mert"
mert_output_dir.mkdir(parents=True, exist_ok=True)
np.save(mert_output_dir / "mert_v1_95m_embeddings.npy", mert_matrix)
mert_index = pd.DataFrame(embedding_rows)
mert_index.to_csv(mert_output_dir / "mert_v1_95m_embedding_index.csv", index=False)
print("Device:", mert_device)
print("Embedding matrix shape:", mert_matrix.shape)
print("Saved outputs to:", mert_output_dir)

## Exploratory clustering of audio windows

Run after the handcrafted-feature and MERT cells. This separately clusters both representations, tests K-means with k=2–6, and saves k=3 labels plus PCA plots. These overlapping windows come from one video and three related stems, so clusters are exploratory acoustic-similarity groups, not semantic classes or independent observations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler, normalize

cluster_dir = MASTER_RESULTS_DIR / "clustering"
cluster_dir.mkdir(parents=True, exist_ok=True)
metadata_columns = ["stem", "start_seconds", "end_seconds"]
feature_columns = [column for column in feature_df.columns if column not in metadata_columns]
if feature_df.empty or not feature_columns:
    raise ValueError("Handcrafted feature rows are empty; run the windowed-features cell first.")
if mert_matrix.shape[0] != len(mert_index):
    raise ValueError("MERT matrix rows do not match its index table.")

mert_metadata = mert_index.sort_values("embedding_row").reset_index(drop=True).rename(columns={"part": "stem"})
keys = ["stem", "start_seconds", "end_seconds"]
feature_metadata = feature_df[keys].copy()
index_metadata = mert_metadata[keys].copy()
feature_metadata["start_seconds"] = feature_metadata["start_seconds"].round(6)
feature_metadata["end_seconds"] = feature_metadata["end_seconds"].round(6)
index_metadata["start_seconds"] = index_metadata["start_seconds"].round(6)
index_metadata["end_seconds"] = index_metadata["end_seconds"].round(6)
aligned_feature_df = mert_metadata[keys].merge(
    feature_df, on=keys, how="left", validate="one_to_one", indicator=True
)
if aligned_feature_df["_merge"].ne("both").any():
    raise ValueError("A MERT window has no matching handcrafted feature row.")
aligned_feature_df = aligned_feature_df.drop(columns="_merge")

representations = {
    "handcrafted_62d": StandardScaler().fit_transform(aligned_feature_df[feature_columns].to_numpy(dtype=float)),
    "mert_768d": normalize(mert_matrix.astype(float), norm="l2"),
}
assignment_frames = []
metric_rows = []
composition_frames = []
markers = {"original_mix": "o", "vocals": "^", "no_vocals": "s"}

for representation_name, matrix in representations.items():
    if not np.isfinite(matrix).all():
        raise ValueError(f"Non-finite values found in {representation_name} vectors.")
    max_k = min(6, len(matrix) - 1)
    for k in range(2, max_k + 1):
        labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(matrix)
        score = float(silhouette_score(matrix, labels))
        metric_rows.append({"representation": representation_name, "k": k, "silhouette": score})
    labels = KMeans(n_clusters=3, random_state=42, n_init=20).fit_predict(matrix)
    coordinates = PCA(n_components=2, random_state=42).fit_transform(matrix)
    if representation_name == "handcrafted_62d":
        metadata = aligned_feature_df[metadata_columns].reset_index(drop=True)
    else:
        metadata = mert_metadata[metadata_columns].reset_index(drop=True)
    assigned = metadata.copy()
    assigned["representation"] = representation_name
    assigned["cluster_k3"] = labels
    assigned["pca_1"] = coordinates[:, 0]
    assigned["pca_2"] = coordinates[:, 1]
    assignment_frames.append(assigned)
    composition = pd.crosstab(assigned["cluster_k3"], assigned["stem"], normalize="index")
    composition = composition.reset_index().melt(id_vars="cluster_k3", var_name="stem", value_name="fraction")
    composition["representation"] = representation_name
    composition_frames.append(composition)

    fig, ax = plt.subplots(figsize=(8, 6))
    for cluster_id in sorted(np.unique(labels)):
        for stem_name in assigned["stem"].unique():
            selected = (assigned["cluster_k3"] == cluster_id) & (assigned["stem"] == stem_name)
            if selected.any():
                ax.scatter(assigned.loc[selected, "pca_1"], assigned.loc[selected, "pca_2"],
                           marker=markers.get(stem_name, "o"), label=f"cluster {cluster_id} / {stem_name}", alpha=0.8)
    ax.set(title=f"{representation_name}: K-means k=3 (PCA view)", xlabel="PCA 1", ylabel="PCA 2")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(cluster_dir / f"{representation_name}_k3_pca.png", dpi=160)
    plt.close(fig)

assignments = pd.concat(assignment_frames, ignore_index=True)
metrics = pd.DataFrame(metric_rows)
composition = pd.concat(composition_frames, ignore_index=True)
assignments.to_csv(cluster_dir / "window_cluster_assignments.csv", index=False)
metrics.to_csv(cluster_dir / "kmeans_silhouette_sweep.csv", index=False)
composition.to_csv(cluster_dir / "k3_cluster_stem_composition.csv", index=False)
print("Silhouette sweep (higher is more compact/separated in this representation; not a semantic score):")
print(metrics.pivot(index="k", columns="representation", values="silhouette").round(3).to_string())
print()
print("k=3 cluster sizes and stem composition:")
print(composition.pivot_table(index=["representation", "cluster_k3"], columns="stem", values="fraction", fill_value=0).round(2).to_string())
print("Saved exploratory cluster outputs to:", cluster_dir)


### Single-stem follow-up: no-vocals only

This isolates the `no_vocals` MERT windows so clusters are not driven by differences between Demucs variants. With only 12 overlapping windows, use the output to explore within-track changes; it is not a reliable semantic classification.

In [ ]:
# Re-cluster only the no_vocals MERT windows as a within-track exploratory check.
no_vocals_mask = mert_index["part"].eq("no_vocals").to_numpy()
no_vocals_metadata = mert_index.loc[no_vocals_mask].sort_values("embedding_row").reset_index(drop=True).copy()
no_vocals_matrix = mert_matrix[no_vocals_mask].astype(float)
if len(no_vocals_matrix) < 3:
    raise ValueError(f"Need at least 3 no_vocals windows; found {len(no_vocals_matrix)}")
no_vocals_matrix = normalize(no_vocals_matrix, norm="l2")
no_vocals_sweep = []
for k in range(2, min(6, len(no_vocals_matrix) - 1) + 1):
    sweep_labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(no_vocals_matrix)
    no_vocals_sweep.append({"k": k, "silhouette": float(silhouette_score(no_vocals_matrix, sweep_labels))})

no_vocals_labels = KMeans(n_clusters=2, random_state=42, n_init=20).fit_predict(no_vocals_matrix)
no_vocals_metadata["cluster_k2"] = no_vocals_labels
no_vocals_out_dir = cluster_dir / "no_vocals_only"
no_vocals_out_dir.mkdir(parents=True, exist_ok=True)
no_vocals_assignments = no_vocals_metadata[[
    "embedding_row", "part", "start_seconds", "end_seconds", "actual_duration_seconds", "cluster_k2"
]]
no_vocals_assignments.to_csv(no_vocals_out_dir / "mert_no_vocals_k2_assignments.csv", index=False)
no_vocals_metrics = pd.DataFrame(no_vocals_sweep)
no_vocals_metrics.to_csv(no_vocals_out_dir / "mert_no_vocals_silhouette_sweep.csv", index=False)
no_vocals_summary = no_vocals_metadata.groupby("cluster_k2").agg(
    window_count=("embedding_row", "size"),
    start_min=("start_seconds", "min"),
    end_max=("end_seconds", "max"),
).reset_index()
no_vocals_summary.to_csv(no_vocals_out_dir / "mert_no_vocals_cluster_summary.csv", index=False)
print("No-vocals-only MERT silhouette sweep:")
print(no_vocals_metrics.round(3).to_string(index=False))
print("\nk=2 time-window assignments:")
print(no_vocals_assignments[["start_seconds", "end_seconds", "cluster_k2"]].to_string(index=False))
print("\nSaved no-vocals-only clustering to:", no_vocals_out_dir)


## Basic Pitch note events and MIDI (optional)

Run in an environment with the Basic Pitch CLI installed. Each available configured audio variant gets its own output folder.

In [ ]:
# Transcribe every configured audio variant with Basic Pitch.
from pathlib import Path
import subprocess

basic_pitch = Path.home() / "venvs/basic-pitch/bin/basic-pitch"
if not basic_pitch.is_file():
    raise FileNotFoundError(f"Basic Pitch CLI not found: {basic_pitch}")

for part, audio_path in files.items():
    if not audio_path.is_file():
        raise FileNotFoundError(f"Audio file for {part!r} not found: {audio_path}")
    part_output_dir = MASTER_RESULTS_DIR / "basic_pitch" / part
    part_output_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        str(basic_pitch), "--save-midi", "--save-note-events",
        str(part_output_dir), str(audio_path),
    ], check=True)
    print(f"{part}: saved Basic Pitch outputs to {part_output_dir}")